# Plot of the 1-D marginal posteriors and coverage for all rounds.

In this notebook, we reproduce Figures 3 to 7 from Pardo et al. (2025). Note that in order to make this notebook work, you first need to download the results data from `/data/magnesia/common/paper_pardo_araujo_etal_2024/experiments_paper.zip`, unpack the file and copy the entire folder experiments into `MAGNESIA_population_synthesis/data/paper_results/pardo_etal_2025/experiments`. The last cell of this notebook generates the posterior distributions for each round of each experiment. To choose a specific experiment to visualize, modify the value of the `experiment_number` variable below.

In [ ]:
import torch
import corner
import matplotlib.pyplot as plt
import os
import numpy as np
import pandas as pd
import json

from matplotlib import rcParams
from matplotlib import rc
import matplotlib as mpl

rc("text", usetex=True)
rc("font", family="serif")
mpl.rcParams["text.latex.preamble"] = r"\usepackage{amsmath}"

In [ ]:
SMALL_SIZE = 30
MEDIUM_SIZE = 50
BIGGER_SIZE = 60

plt.rc("font", size=SMALL_SIZE)  
plt.rc("axes", titlesize=MEDIUM_SIZE)  
plt.rc("axes", labelsize=MEDIUM_SIZE)  
plt.rc("xtick", labelsize=SMALL_SIZE)  
plt.rc("ytick", labelsize=SMALL_SIZE)  
plt.rc("legend", fontsize=SMALL_SIZE)  
plt.rc("figure", titlesize=MEDIUM_SIZE)  

In [ ]:
def import_statistics(stats_path: str):
    """
    Extracting the mean and standard deviation for all the parameters in the `stats_path` file.
    Args:
        stats_path (str): Path to the file where the statistics are saved.
    Returns:
        (torch.tensor, torch.tensor): Mean and standard deviation for the parameters in the `stats_path` file.
    """
    std_list = []
    mean_list = []
    max_list = []
    min_list = []
    with open(stats_path, "r") as json_file:
        data = json.load(json_file)
    for key, value in data.items():
        std_list.append(value["std"])
        mean_list.append(value["mean"])
        max_list.append(value["max"])
        min_list.append(value["min"])
    mean = np.array(mean_list)
    std = np.array(std_list)
    max_list = np.array(max_list)
    min_list = np.array(min_list)
    return mean, std, max_list, min_list

Chose the experiment and rounds to be plotted.

In [ ]:
experiment_number = 3
n_rounds = 10

In [ ]:
# Loading results of experiments and the statitsics of the training datasets. 
directory_path = f"../../data/paper_results/pardo_etal_2025/experiments/exp_{experiment_number}"
stats_path = directory_path+"/statistics_train.json"

In [ ]:
true_values = [13.25,0.75,-0.6,0.3,-2,26.9,0.5]

observed_posterior_round = []
coverage_round = []

for i in range(n_rounds):
    observed_posterior_round.append(torch.load(f"{directory_path}/round_{i}/samples_posterior_{i}.pt").detach().cpu().numpy())
    coverage_round.append(np.load(f"{directory_path}/inference/round_{i}/coverage_probability.npy"))

Loading the results of Graber et al 2023 to reproduce Figure 3 and 4, where we compare the results of this work with our previous results.

In [ ]:
exp_number = [
    1,
    2,
    3,
    4,
    5,
    6,
    8,
    9,
    10,
    11,
    12,
    13,
    14,
    15,
    16,
    17,
    19,
    20,
    21,
]

In [ ]:
posteriors_atnf = []

for n in exp_number:
    
    posteriors_atnf.append(
        torch.load(f"../../data/paper_results/graber_etal_2024/samples/samples_exp_{n}_atnf.pt")
        .detach()
        .cpu()
        .numpy()
        .T
    )
posteriors_ensemble_atnf = np.concatenate(posteriors_atnf, axis=1)

In [ ]:
mean, std, par_max, par_min = import_statistics(stats_path)

In [ ]:
limits = [[12.0, 14.0], [0.1, 1.0], [-1.5, -0.3], [0.1, 1.0], [-3, -0.5],[24.6,28.6],[0.1,1]]
x_ticks = [[12.0,13.0, 14.0], [0.1,0.5, 0.9], [-1.5,-1, -0.5], [0.1, 0.5,0.9], [-3, -2,-1],[24.6,26.6,28.5],[0.2,0.5,1]]
parameter_labels = [
    r"$\mu_{\log B}$",
    r"$\sigma_{\log B}$",
    r"$\mu_{\log P}$",
    r"$\sigma_{\log P}$",
    r"$a_{\rm late}$",
    r"$\mu_{\log L_0}$",
    r"$\alpha$"
]

n_param = np.shape(observed_posterior_round)[2]

In [ ]:
credibility_level = np.linspace(0, 1, 12)

In [ ]:
n_param = np.shape(observed_posterior_round[0])[1]

fig, axs = plt.subplots(n_rounds, n_param+1, figsize=(27, 15),gridspec_kw={'hspace': 0, 'wspace': 0.2})
for j in range(n_param):
    for i in range(n_rounds):
        observed_posterior = observed_posterior_round[i]
        observed_posterior = observed_posterior * std[0:n_param] + mean[0:n_param]

        # Set the color for each round posterior.
        curve_color = 'tab:blue'  
        
        # Select the correct axis for each round and parameter.
        ax = axs[i][j]
        
        # Calculate the 95% confidence interval using percentiles to plotted is a grey shaded area.
        lower_bound = np.percentile(observed_posterior.T[j], 2.5)
        upper_bound = np.percentile(observed_posterior.T[j], 97.5)

        # Add grey shading for the 95% confidence interval
        ax.axvspan(lower_bound, upper_bound, color='tab:gray', alpha=0.3)

        # If the number of parameter is equal to 5 plot also the results from Graber et al 2024.
        if n_param ==5:
            ax.hist(
            posteriors_ensemble_atnf[j],
            bins=32,
            color='black',
            histtype="step",
            linewidth=2,
            density=True,
        )
        
        ax.hist(
            observed_posterior.T[j],
            bins=32,
            color=curve_color,
            edgecolor=curve_color,
            linewidth=2,
            histtype="step",
            density=True,
        )
        
        for past_round in range(0,i):
            observed_posterior_past_round = observed_posterior_round[past_round]
            observed_posterior_past_round = observed_posterior_past_round * std[0:n_param] + mean[0:n_param]

            ax.hist(
                observed_posterior_past_round.T[j],
                bins=32,
                color='grey',
                edgecolor='grey',
                alpha = 0.5,
                linewidth=1.5,
                histtype="step",
                density=True,
            )
        if experiment_number == 3:
            # Add vertical line for ground truths if we are plotting Experiment 3.
            ax.axvline(
                true_values[j],
                color='tab:orange',  
                linestyle='--',  
                linewidth=2, 
            )
        
        # Only add x-axis tick marks (without labels) for all rows except the last.
        if i < n_rounds - 1 and i>0:
            ax.tick_params(axis='x', which='major', length=10, width=1.5,top=True, labeltop=False, bottom=True, labelbottom=False, direction = 'inout')  # For major ticks
            ax.tick_params(axis='x', which='minor', length=6, width=1,top=True, labeltop=False, bottom=True, labelbottom=False, direction = 'inout')  
            ax.set_xticks(x_ticks[j])
            ax.minorticks_on()
        elif i == 0:
            ax.tick_params(axis='x', which='major', length=10, width=1.5,top=False, labeltop=False, bottom=True, labelbottom=False, direction = 'inout')  # For major ticks
            ax.tick_params(axis='x', which='minor', length=6, width=1,top=False, labeltop=False, bottom=True, labelbottom=False, direction = 'inout',grid_color='r', grid_alpha=0.5)  
            ax.set_xticks(x_ticks[j])
            ax.minorticks_on()
        else:
            ax.tick_params(axis='x', which='major', length=10, width=1.5,top=True, labeltop=False, bottom=True, labelbottom=True, direction = 'inout')  # For major ticks
            ax.tick_params(axis='x', which='minor', length=6, width=1,top=True, labeltop=False, bottom=True, labelbottom=True, direction = 'inout')  
            ax.set_xlabel(parameter_labels[j], fontsize=MEDIUM_SIZE)
            ax.set_xticks(x_ticks[j])
            ax.minorticks_on()
            
            
        # Remove y-axis ticks for all subplots.
        ax.set_yticks([])
        
        # Only set titles in the top row.
        if i == 0:
            ax.set_title(parameter_labels[j], fontsize=MEDIUM_SIZE)

        # Add the round number next to each row, rotated parallel to the y-axis.
        fig.text(0.12, 0.83 - (i / 1.28 - 0.18) / n_rounds, f'Round {i+1}', va='center', ha='center',
                 fontsize=18, rotation='vertical')

        # Set limits for the x-axis
        ax.set_xlim(limits[j])
        
        # Automagically incresing the ylimit axis to a 30% more than the data described.
        ax.margins(y=0.3)
           
# Plot the coverage probability in the last column.
for i in range(n_rounds):
    ax = axs[i][n_param]  
    for past_round in range(0,i):
        ax.plot(
            credibility_level,
            coverage_round[past_round],
            linestyle="-",
            color='tab:grey',
            linewidth=1,
            alpha=0.3,
            rasterized=True
        )
    ax.plot(
            credibility_level,
            coverage_round[i],
            linestyle="-",
            color='tab:blue',
            linewidth=3,
            rasterized=True
        )
    ax.plot(
        credibility_level,
        credibility_level,
        linestyle="-",
        color="black",
        linewidth=2,
        alpha=1,
        rasterized=True,
        label=r"Well-calibrated",
    )
    
    coverage_ticks = [0.1,0.5,0.9]
    ax.set_xlim(0.01,0.99)
    ax.set_ylim(0.01,0.99)

    if i < n_rounds - 1 and i>0:
        ax.grid(which='both')
        ax.tick_params(axis='x', which='major', length=10, width=1.5,top=True, labeltop=False, bottom=True, labelbottom=False, direction = 'inout')  # For major ticks
        ax.set_xticks(coverage_ticks)
        

    elif i == 0:
        ax.grid(which='both')
        ax.tick_params(axis='x', which='major', length=10, width=1.5, top=False, labeltop=False, bottom=True, labelbottom=False)
        ax.set_xticks(coverage_ticks)

    else:
        ax.grid(which='both')
        ax.set_xlabel(r"Credibility level $1 - \alpha$", fontsize=SMALL_SIZE)
        ax.set_xticks(coverage_ticks)

    if i == 0:
        ax.set_title(r"Credibility level $1 - \alpha$", fontsize=SMALL_SIZE)

    credibility_ticks = [0.1,0.5,0.9]
    ax.set_yticks(credibility_ticks,credibility_ticks, fontsize = SMALL_SIZE-4)
    ax.tick_params(axis='y', which='major', length=10, width=1.5,left=False, labelleft=False, right=False, labelright=False, direction = 'out')  # For major ticks
    
    if i == n_rounds-1:
        ax.tick_params(axis='y', which='major', length=10, width=1.5,left=False, labelleft=False, right=True, labelright=True, direction = 'out')  # For major ticks
        ax.set_yticks(credibility_ticks,credibility_ticks, fontsize = SMALL_SIZE)
        
fig.text(0.91, 0.5, 'Coverage Probability', va='center', ha='center', fontsize=SMALL_SIZE, rotation=270)

plt.savefig(f'../../paper_plots/pardo_et_al_2025/plots/exp_{experiment_number}.pdf',bbox_inches="tight")

plt.show()